# **Crop Type Classification from Sentinel-2 Time Series**

EuroCropsML combines EuroCrops (v9) parcel data with 2021 Sentinel-2 reflectance data from Latvia, Portugal, and Estonia for few-shot crop classification benchmarking


## Installations

In [ ]:
# !pip install -q huggingface_hub

## Imports

In [ ]:
from huggingface_hub import hf_hub_download
import zipfile, io
import numpy as np
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd

## Helper Functions


In [ ]:
FILES = {
    "preprocess": "preprocess/portugal.zip",
    "raw": "raw_data/Portugal.parquet",
    "labels": "raw_data/labels/Portugal_labels.parquet",
    "geometry": "raw_data/geometries/Portugal.geojson",
    "classes": "cropclasses.csv",
}

def fetch_data(file_name, data_dir="data"):
    path = Path(data_dir) / FILES[file_name]

    if not path.exists():
      path = Path(
          hf_hub_download(
              repo_id="tbbotchwey/eurocropsml",
              filename=FILES[file_name],
              repo_type="dataset",
              local_dir=data_dir,
          )
        )
    if file_name == "preprocess":
      return zipfile.ZipFile(path)
    if file_name == "geometry":
      return gpd.read_file(path)
    if file_name == "classes":
      return pd.read_csv(path)
    return pd.read_parquet(path)


# GLOBAL FILENAMES
preprocess = fetch_data("preprocess")

labels  = fetch_data("labels")
classes = fetch_data("classes")

## Extract Preprocess Data

Flatten every `.npz` in the preprocess zip into one table with one row per parcel and observation date.

Each file is named `{NUTS3}_{parcel_id}_{HCAT code}.npz` and holds:
- `data`: dates × 13 Sentinel-2 L1C bands
- `dates`: observation dates
- `center`: parcel midpoint (lon, lat)


In [ ]:
BANDS = ["B01", "B02", "B03", "B04", "B05", "B06", "B07",
         "B08", "B8A", "B09", "B10", "B11", "B12"]

npz_files = sorted(n for n in preprocess.namelist() if n.endswith(".npz"))

parcels, dates, values = [], [], []
for name in npz_files:
    file_name = Path(name).name
    nuts3, parcel_id, hcat_code = file_name[:-4].split("_")
    with np.load(io.BytesIO(preprocess.read(name))) as npz:
        lon, lat = npz["center"]
        parcels.append((int(parcel_id), file_name, lon, lat, int(hcat_code), nuts3, len(npz["dates"])))
        dates.append(npz["dates"])
        values.append(npz["data"].astype(np.int16))

parcels = pd.DataFrame(parcels, columns=["parcel_id", "file_name", "midpoint_lon", "midpoint_lat",
                                         "hcat_crop_code", "nuts3", "n_observations"])
print(f"{len(parcels):,} parcels, {parcels['n_observations'].sum():,} observations")


In [ ]:
parcels

Attach crop names from `labels` and the coarser HCAT level-3 class from `classes`.

In [ ]:
crop_labels = labels.drop_duplicates("parcel_id").set_index("parcel_id")
level3 = classes.set_index("HCAT3_level6_code")

parcels["hcat_crop_name"] = parcels["parcel_id"].map(crop_labels["EC_hcat_n"])
parcels["hcat_level3_code"] = parcels["hcat_crop_code"].map(level3["HCAT3_level3_code"])
parcels["hcat_level3_name"] = parcels["hcat_crop_code"].map(level3["HCAT3_level3_name"])

# check that the HCAT code in the file name agrees with the labels file
label_code = parcels["parcel_id"].map(crop_labels["EC_hcat_c"]).astype("int64")
print("code mismatches:", (label_code != parcels["hcat_crop_code"]).sum())
print("missing values:\n", parcels.isna().sum())


Expand to one row per observation date and add the band values.

In [ ]:
df = parcels.loc[parcels.index.repeat(parcels["n_observations"])].reset_index(drop=True)
df["date"] = np.concatenate(dates).astype("datetime64[D]")
df["day_of_year"] = df["date"].dt.dayofyear
df = pd.concat([df, pd.DataFrame(np.concatenate(values), columns=BANDS)], axis=1)

df = df[["parcel_id", "file_name", "date", "day_of_year", "midpoint_lon", "midpoint_lat",
         "hcat_crop_code", "hcat_crop_name", "hcat_level3_code", "hcat_level3_name",
         "nuts3", "n_observations", *BANDS]]
print(df.shape)
df.head()


In [ ]:
out_path = Path("output/portugal_preprocess_long.parquet")
out_path.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(out_path, index=False)
print(f"Saved {out_path}")


## Filter to Kept Parcels

In [ ]:
import ast


with open("./kept_files3.txt", "r") as f:
    kept = ast.literal_eval(f.read())

print(len(kept))
print(type(kept))


In [ ]:
df_kept = df[df["file_name"].isin(kept)].reset_index(drop=True)
df_kept["day_of_year"] = df_kept["date"].dt.dayofyear
print("kept files not found in df:", len(set(kept) - set(df["file_name"])))
print(f"{df_kept['parcel_id'].nunique():,} of {df['parcel_id'].nunique():,} parcels, {len(df_kept):,} rows")
df_kept.head()


In [ ]:
df_kept.isna().sum()

In [ ]:
# 0 in the band columns
zeros = (df_kept.select_dtypes("number") == 0).sum()
print(f"{(zeros > 0).sum()} of {df_kept.shape[1]} columns contain 0s")
zeros[zeros > 0]

In [ ]:
df_kept_clean = df_kept[df_kept["hcat_crop_name"] != "not_known_and_other"].reset_index(drop=True)
print(df_kept.shape)
print(df_kept_clean.shape)

In [ ]:
diff = df_kept.shape[0] - df_kept_clean.shape[0]
pct = diff / df_kept.shape[0]

diff, pct

In [ ]:
# 0 in the band columns
zeros = (df_kept_clean.select_dtypes("number") == 0).sum()
print(f"{(zeros > 0).sum()} of {df_kept_clean.shape[1]} columns contain 0s")
zeros[zeros > 0]

In [ ]:
df_kept_clean_non_zero = df_kept_clean[
    (df_kept_clean["B02"] != 0) &
    (df_kept_clean["B03"] != 0) &
    (df_kept_clean["B04"] != 0) &
    (df_kept_clean["B05"] != 0) &
    (df_kept_clean["B06"] != 0) &
    (df_kept_clean["B07"] != 0) &
    (df_kept_clean["B08"] != 0) &
    (df_kept_clean["B8A"] != 0) &
    (df_kept_clean["B11"] != 0) &
    (df_kept_clean["B12"] != 0)
    # (df_kept_clean["B08"] == 0) 
]

In [ ]:
# 0 in the band columns
zeros = (df_kept_clean_non_zero.select_dtypes("number") == 0).sum()
print(f"{(zeros > 0).sum()} of {df_kept_clean_non_zero.shape[1]} columns contain 0s")
zeros[zeros > 0]

In [ ]:
df_kept_clean_non_zero_rgbn = df_kept_clean_non_zero.drop(columns=["B01", "B09", "B10"], errors="ignore")

In [ ]:
df_kept_clean_non_zero_rgbn

In [ ]:
# 0 in the band columns
rem = df_kept_clean_non_zero_rgbn.shape[0]
tot = df_kept.shape[0]
print(f"{rem} data points are remaining of {tot}\nThat is {tot - rem} ({(tot - rem)/tot})")
zeros = (df_kept_clean_non_zero_rgbn.select_dtypes("number") == 0).sum()
print(f"{(zeros > 0).sum()} of {df_kept_clean_non_zero_rgbn.shape[1]} columns contain 0s")

In [ ]:
kept_path = Path("output/portugal_preprocess_kept.parquet")
df_kept.to_parquet(kept_path, index=False)
print(f"Saved {kept_path}")


In [ ]:
df_kept

## Resample to 15-day Periods

One row per parcel and 15-day period that has data (24 periods. days 361-365 are folded into the last one).
Band values are the **median** of the observations in each period:
- clouds are already masked out in the dataset, so mean and median give very similar results
- with 1-2 observations (the most common case) the median equals the mean
- with 3+ observations the median ignores a single outlier the cloud mask missed (e.g. cloud shadow or haze), which the mean would average in

Rows with zeros in the band columns are removed before aggregating


In [ ]:
df_kept_clean_non_zero_rgbn.columns

In [ ]:
df_kept_clean_non_zero_rgbn.columns.tolist()

In [ ]:
PERIOD_DAYS = 15
N_PERIODS = 24  # days 361-365 are folded into the last period
BANDS_ = ['B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B11', 'B12']
# BANDS_ = ["B02", "B03", "B04", "B08"]

ts = df_kept_clean_non_zero_rgbn.copy()
# ts[BANDS_] = ts[BANDS_].replace(0, np.nan)  # 0 means no data, so it must not pull the median down
ts["period"] = np.minimum((ts["day_of_year"] - 1) // PERIOD_DAYS, N_PERIODS - 1)

grouped = ts.groupby(["parcel_id", "period"])
df_15d = grouped[BANDS_].median()
df_15d["n_obs_in_period"] = grouped.size()

# keep only periods with at least one observation
df_15d = df_15d.reset_index()
df_15d["n_obs_in_period"] = df_15d["n_obs_in_period"].astype("int16")
df_15d["period_start_doy"] = df_15d["period"] * PERIOD_DAYS + 1
df_15d["period_start"] = pd.Timestamp("2021-01-01") + pd.to_timedelta(df_15d["period_start_doy"] - 1, unit="D")

parcel_info = ts.drop_duplicates("parcel_id").set_index("parcel_id")[
    ["file_name", "midpoint_lon", "midpoint_lat", "hcat_crop_code", "hcat_crop_name",
     "hcat_level3_code", "hcat_level3_name", "nuts3", "n_observations"]]
df_15d = df_15d.join(parcel_info, on="parcel_id")

df_15d = df_15d[["parcel_id", "file_name", "period", "period_start", "period_start_doy",
                 "midpoint_lon", "midpoint_lat", "hcat_crop_code", "hcat_crop_name",
                 "hcat_level3_code", "hcat_level3_name", "nuts3", "n_observations",
                 "n_obs_in_period", *BANDS_]]
print(df_15d.shape)

# NEW
periods_per_parcel = df_15d.groupby("parcel_id").size()
print(f"{df_15d['parcel_id'].nunique():,} parcels, {periods_per_parcel.min()}-{periods_per_parcel.max()} periods each")
print(f"parcels with all {N_PERIODS} periods: {(periods_per_parcel == N_PERIODS).sum():,}")
print("periods with no observation:", (df_15d["n_obs_in_period"] == 0).sum())


In [ ]:
df_15d

In [ ]:
df_15d.isna().sum()

In [ ]:
path_15d = Path("output/portugal_kept_15day_median.parquet")
df_15d.to_parquet(path_15d, index=False)
print(f"Saved {path_15d}")


In [ ]:
import random

idx = random.choice(range(1, len(npz_files)))

dummy = np.load(io.BytesIO(preprocess.read(npz_files[idx])))
dummy["data"]